### RAG Citation: LLM-Based Citation Generation
#### 5. Example — LLM Method (via LiteLLM)

This notebook demonstrates the **LLM-based citation method** introduced in `rag-citation` v0.1.0.

Instead of the algorithmic SpaCy + SentenceTransformers pipeline, the LLM method asks any LLM
(OpenAI, Anthropic, Azure, Gemini, …) to identify which source document supports each sentence
in the generated answer. It uses **LiteLLM** for provider-agnostic calls and **Pydantic structured
output** to guarantee a well-formed response.

**When to use the LLM method vs the Non-LLM method:**

| | Non-LLM (default) | LLM method |
|---|---|---|
| Speed | Fast (no API call) | Slower (LLM round-trip) |
| Cost | Free | Depends on model |
| Accuracy | Good for entity-rich answers | Better for paraphrased / complex answers |
| Hallucination detection | Yes (entity-based) | No |
| Requires API key | No | Yes |

In [ ]:
# Install the package with LLM extras
# !pip install rag-citation[llm]

## Setup: documents and answer

In [2]:
import uuid

## Context retrieved from a vector DB / semantic search
documents = [
    "Elon Musk Elon MuskCEO, Tesla$221.6B$439M (0.20%)Real Time Net Worthas of 8/6/24Reflects change since 5 pm ET of prior trading day. 1 in the world todayPhoto by Martin Schoeller for ForbesAbout Elon MuskElon Musk cofounded six companies, including electric car maker Tesla, rocket producer SpaceX and tunneling startup Boring Company.He owns about 12% of Tesla excluding options, but has pledged more than half his shares as collateral for personal loans of up to $3.5 billion.In early 2024, a Delaware judge voided Musk's 2018 deal to receive options equaling an additional 9% of Tesla. Forbes has discounted the options by 50% pending Musk's appeal.SpaceX, founded in 2002, is worth nearly $180 billion after a December 2023 tender offer of up to $750 million; SpaceX stock has quintupled its value in four years.Musk bought Twitter in 2022 for $44 billion, after later trying to back out of the deal. He owns an estimated 74% of the company, now called X.Forbes estimates that Musk's stake in X is now worth nearly 70% less than he paid for it based on investor Fidelity's valuation of the company as of December 2023.",
    "people in the world; as of August 2024[update], Forbes estimates his net worth to be US$241 billion.[3] Musk was born in Pretoria to model Maye and businessman and engineer Errol Musk, and briefly attended the University of Pretoria before immigrating to Canada at age 18, acquiring citizenship through his Canadian-born mother. Two years later, he matriculated at Queen's University at Kingston in Canada. Musk later transferred to the University of Pennsylvania and received bachelor's degrees in economics and physics.",
]

## Answer generated by an LLM
answer = "Elon Musk's net worth is estimated to be US$241 billion as of August 2024."

## Build context in the format CiteItem expects
context = [
    {
        "source_id": str(uuid.uuid4()),
        "document": doc,
        "meta": [{"url": "https://www.forbes.com/profile/elon-musk/", "chunk_id": str(uuid.uuid4())}],
    }
    for doc in documents
]

## Example 1 — Basic LLM citation

Set `method="llm"` and provide a `model` name in [LiteLLM format](https://docs.litellm.ai/docs/providers).
The API key is read from the environment variable automatically (`OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc.),
or you can pass it explicitly via `api_key=`.

In [21]:
api_key="sk-****"

In [11]:
from rag_citation import CiteItem, Inference

cite_item = CiteItem(answer=answer, context=context)

inference = Inference(
    method="llm",
    model="gpt-4o",        # any LiteLLM-supported model
    api_key=api_key,   # optional — reads OPENAI_API_KEY from env if not set
)

output = inference(cite_item)

In [13]:
output.citation

[{'answer_sentences': "Elon Musk's net worth is estimated to be US$241 billion as of August 2024.",
  'cite_document': [{'document': "people in the world; as of August 2024[update], Forbes estimates his net worth to be US$241 billion.[3] Musk was born in Pretoria to model Maye and businessman and engineer Errol Musk, and briefly attended the University of Pretoria before immigrating to Canada at age 18, acquiring citizenship through his Canadian-born mother. Two years later, he matriculated at Queen's University at Kingston in Canada. Musk later transferred to the University of Pennsylvania and received bachelor's degrees in economics and physics.",
    'source_id': '840ac3dd-a507-4a90-8d52-72c33ac0153f',
    'score': None,
    'entity': [],
    'meta': [{'url': 'https://www.forbes.com/profile/elon-musk/',
      'chunk_id': '4df9e045-09bd-47eb-96ed-f49f529e1fd8'}]}]}]

Expected output:
```python
[{
    'answer_sentences': "Elon Musk's net worth is estimated to be US$241 billion as of August 2024.",
    'cite_document': [{
        'document': 'people in the world; as of August 2024[update], Forbes estimates his net worth to be US$241 billion...',
        'source_id': '<uuid>',
        'score': None,
        'entity': [],
        'meta': [{'url': 'https://www.forbes.com/profile/elon-musk/', 'chunk_id': '<uuid>'}]
    }]
}]
```

> **Note:** `score` and `entity` are `None`/`[]` for the LLM method — semantic scoring and NER
> are only performed by the non-LLM pipeline. `missing_word` and `hallucination` are also always
> `[]` / `False` for the LLM method.

In [14]:
print("missing_word :", output.missing_word)
print("hallucination:", output.hallucination)

missing_word : []
hallucination: False


## Example 2 — Pass conversation history for richer context

If you have the full conversation that produced the answer you can pass it via `messages=`.
The citation instructions are appended to the existing conversation, giving the LLM more
context to make accurate attribution decisions.

In [18]:
from rag_citation import CiteItem, Inference

cite_item = CiteItem(answer=answer, context=context)

inference = Inference(method="llm", model="gpt-4o",api_key=api_key)

## The conversation that produced the answer
messages = [
    {"role": "user", "content": "What is Elon Musk's net worth?"},
    {"role": "assistant", "content": answer},
]

output = inference(cite_item, messages=messages)
output.citation

[{'answer_sentences': "Elon Musk's net worth is estimated to be US$241 billion as of August 2024.",
  'cite_document': [{'document': "people in the world; as of August 2024[update], Forbes estimates his net worth to be US$241 billion.[3] Musk was born in Pretoria to model Maye and businessman and engineer Errol Musk, and briefly attended the University of Pretoria before immigrating to Canada at age 18, acquiring citizenship through his Canadian-born mother. Two years later, he matriculated at Queen's University at Kingston in Canada. Musk later transferred to the University of Pennsylvania and received bachelor's degrees in economics and physics.",
    'source_id': '840ac3dd-a507-4a90-8d52-72c33ac0153f',
    'score': None,
    'entity': [],
    'meta': [{'url': 'https://www.forbes.com/profile/elon-musk/',
      'chunk_id': '4df9e045-09bd-47eb-96ed-f49f529e1fd8'}]}]}]

## Example 3 — Custom system prompt

You can override the default citation system prompt by patching `rag_citation.llm.prompt.CITATION_SYSTEM_PROMPT`
before creating the `Inference` instance.

In [19]:
import rag_citation.llm.prompt as prompt_module

## Strict prompt: only cite when there is a direct factual match
prompt_module.CITATION_SYSTEM_PROMPT = """You are a strict citation assistant.
For each sentence in the answer:
1. Only cite a sentence if the source document contains the EXACT same facts (numbers, dates, names).
2. If a sentence combines information from multiple sources, cite all relevant source documents.
3. If a sentence is not directly supported by any source, do NOT include it in the citations."""

from rag_citation import CiteItem, Inference

cite_item = CiteItem(answer=answer, context=context)
inference = Inference(method="llm", model="gpt-4o",api_key=api_key)
output = inference(cite_item)
output.citation

[{'answer_sentences': "Elon Musk's net worth is estimated to be US$241 billion as of August 2024.",
  'cite_document': [{'document': "people in the world; as of August 2024[update], Forbes estimates his net worth to be US$241 billion.[3] Musk was born in Pretoria to model Maye and businessman and engineer Errol Musk, and briefly attended the University of Pretoria before immigrating to Canada at age 18, acquiring citizenship through his Canadian-born mother. Two years later, he matriculated at Queen's University at Kingston in Canada. Musk later transferred to the University of Pennsylvania and received bachelor's degrees in economics and physics.",
    'source_id': '840ac3dd-a507-4a90-8d52-72c33ac0153f',
    'score': None,
    'entity': [],
    'meta': [{'url': 'https://www.forbes.com/profile/elon-musk/',
      'chunk_id': '4df9e045-09bd-47eb-96ed-f49f529e1fd8'}]}]}]

## Example 4 — Using other providers via LiteLLM

LiteLLM supports 100+ providers. Just change the `model` string.

In [ ]:
from rag_citation import CiteItem, Inference

cite_item = CiteItem(answer=answer, context=context)

## --- Anthropic ---
# inference = Inference(method="llm", model="anthropic/claude-sonnet-4-20250514")

## --- Azure OpenAI ---
# inference = Inference(
#     method="llm",
#     model="azure/gpt-4o",
#     api_key="your-azure-key",
#     api_base="https://your-resource.openai.azure.com",
#     api_version="2024-02-01",
# )

## --- Google Gemini ---
# inference = Inference(method="llm", model="gemini/gemini-1.5-pro")

## Uncomment one of the above and run:
# output = inference(cite_item)
# output.citation

## Comparing Non-LLM vs LLM output on the same input

In [20]:
from rag_citation import CiteItem, Inference

cite_item = CiteItem(answer=answer, context=context)

## Non-LLM
inference_nonllm = Inference(spacy_model="sm", embedding_model="md")
output_nonllm = inference_nonllm(cite_item)

## LLM
inference_llm = Inference(method="llm", model="gpt-4o",api_key=api_key)
output_llm = inference_llm(cite_item)

print("=== Non-LLM citation ===")
for c in output_nonllm.citation:
    print("sentence  :", c["answer_sentences"])
    for d in c["cite_document"]:
        print("  source_id:", d["source_id"])
        print("  score    :", d["score"])
        print("  entity   :", d["entity"])

print()
print("=== LLM citation ===")
for c in output_llm.citation:
    print("sentence  :", c["answer_sentences"])
    for d in c["cite_document"]:
        print("  source_id:", d["source_id"])
        print("  score    :", d["score"])   # None for LLM method
        print("  entity   :", d["entity"])  # [] for LLM method

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

100%|██████████| 1/1 [00:01<00:00,  1.11s/it]


=== Non-LLM citation ===
sentence  : Elon Musk's net worth is estimated to be US$241 billion as of August 2024.
  source_id: 840ac3dd-a507-4a90-8d52-72c33ac0153f
  score    : 100
  entity   : [{'word': 'August 2024', 'entity_name': 'DATE'}, {'word': 'US$241 billion', 'entity_name': 'MONEY'}]

=== LLM citation ===
sentence  : Elon Musk's net worth is estimated to be US$241 billion as of August 2024.
  source_id: 840ac3dd-a507-4a90-8d52-72c33ac0153f
  score    : None
  entity   : []
